In [21]:

import numpy as np
import h5py

In [22]:
import numpy as np
import h5py

class Dataset_Handling:

    def __init__(self):
        pass

    def load_dataset(self):
        train_dataset = h5py.File('train_catvnoncat.h5', "r")
        train_set_x_orig = np.array(train_dataset["train_set_x"][:])
        train_set_y_orig = np.array(train_dataset["train_set_y"][:])

        test_dataset = h5py.File('test_catvnoncat.h5', "r")
        test_set_x_orig = np.array(test_dataset["test_set_x"][:])
        test_set_y_orig = np.array(test_dataset["test_set_y"][:])

        classes = np.array(test_dataset["list_classes"][:])

        train_set_y_orig = train_set_y_orig.reshape((1, train_set_y_orig.shape[0]))
        test_set_y_orig = test_set_y_orig.reshape((1, test_set_y_orig.shape[0]))

        return train_set_x_orig, train_set_y_orig, test_set_x_orig, test_set_y_orig, classes


    def re_shape(self):

        X_train, Y_train, X_test, Y_test, classes = self.load_dataset()

        # flatten images
        X_train = X_train.reshape(X_train.shape[0], -1).T
        X_test = X_test.reshape(X_test.shape[0], -1).T
      
        # normalize
        X_train = X_train / 255.
        X_test = X_test / 255.

        # number of examples
        m_train = X_train.shape[1]
        m_test = X_test.shape[1]

        # add row of ones for theta0
        ones_train = np.ones((1, m_train))
        ones_test = np.ones((1, m_test))

        X_train = np.vstack((ones_train, X_train))
        X_test = np.vstack((ones_test, X_test))

        return X_train, Y_train, X_test, Y_test, classes

In [23]:
obj=Dataset_Handling()
X_train, Y_train, X_test, Y_test, classes=obj.re_shape()

In [24]:

import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

class LogisticRegression:

    def __init__(self, lr=0.01, epochs=2000):
        self.lr = lr
        self.epochs = epochs
        self.theta = None
        self.scaler = StandardScaler()

 
    def scale_features(self, X, fit=False):

        bias = X[0:1, :]          # keep bias row
        features = X[1:, :]       # actual features

        features = features.T     # (examples, features)

        if fit:
            features = self.scaler.fit_transform(features)
        else:
            features = self.scaler.transform(features)

        features = features.T

        X_scaled = np.vstack((bias, features))

        return X_scaled


  
    def sigmoid(self, z):

        return 1 / (1 + np.exp(-z))


   
    def compute_loss(self, A, Y):

        m = Y.shape[1]

        loss = -(1/m) * np.sum(Y*np.log(A + 1e-8) + (1-Y)*np.log(1-A + 1e-8))

        return loss





   
    def fit(self, X, Y):

        n = X.shape[0]
        m = X.shape[1]

        self.theta = np.zeros((n,1))

        for i in range(self.epochs):

            Z = np.dot(self.theta.T, X)
            A = self.sigmoid(Z)

            loss = self.compute_loss(A, Y)

            dtheta = (1/m) * np.dot(X, (A-Y).T)

            self.theta = self.theta - self.lr * dtheta

            if i % 100 == 0:
                print("Iteration", i, "Loss:", loss)


    
    def predict(self, X):

        Z = np.dot(self.theta.T, X)
        A = self.sigmoid(Z)

        return (A > 0.5).astype(int)


   
    def accuracy(self, X, Y):

        pred = self.predict(X)

        return 100 - np.mean(np.abs(pred - Y))*100
    
    def confusion_matrix(self,X,Y):

        pred = self.predict(X)

        TP = np.sum((pred==1) & (Y==1))
        TN = np.sum((pred==0) & (Y==0))
        FP = np.sum((pred==1) & (Y==0))
        FN = np.sum((pred==0) & (Y==1))

        return TP,TN,FP,FN
    
    

In [25]:
model = LogisticRegression(lr=0.01, epochs=3000)


# X_train = model.scale_features(X_train, fit=True)
# X_test = model.scale_features(X_test, fit=False)


model.fit(X_train, Y_train)


print("Train Accuracy:", model.accuracy(X_train, Y_train))
print("Test Accuracy:", model.accuracy(X_test, Y_test))



TP,TN,FP,FN = model.confusion_matrix(X_train,Y_train)

print("TP:",TP)
print("TN:",TN)
print("FP:",FP)
print("FN:",FN)


Iteration 0 Loss: 0.6931471605599454
Iteration 100 Loss: 0.8239208323545103
Iteration 200 Loss: 0.41894441899973117
Iteration 300 Loss: 0.6173496666898499
Iteration 400 Loss: 0.5221157382629192
Iteration 500 Loss: 0.3877087258798497
Iteration 600 Loss: 0.23625444220008374
Iteration 700 Loss: 0.15422212100243227
Iteration 800 Loss: 0.1353278166380101
Iteration 900 Loss: 0.12497146850178142
Iteration 1000 Loss: 0.11647831988545632
Iteration 1100 Loss: 0.1091925000147561
Iteration 1200 Loss: 0.10280445300257328
Iteration 1300 Loss: 0.09712979897529712
Iteration 1400 Loss: 0.09204325819778841
Iteration 1500 Loss: 0.08745250893746469
Iteration 1600 Loss: 0.08328601960836204
Iteration 1700 Loss: 0.07948655949714449
Iteration 1800 Loss: 0.07600733488243568
Iteration 1900 Loss: 0.07280948378550822
Iteration 2000 Loss: 0.06986034528593883
Iteration 2100 Loss: 0.06713220723013023
Iteration 2200 Loss: 0.06460136884571586
Iteration 2300 Loss: 0.06224742113887545
Iteration 2400 Loss: 0.060052683718